# ToolStrategy 的多种用法

输出模式2：TypedDict类型

In [ ]:
from anthropic import BaseModel
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from typing import TypedDict, Annotated
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-chat"
)


class ContactInfo(TypedDict):
    """用户的联系方式"""
    name: Annotated[str, ..., "用户姓名"]
    email: Annotated[str, ..., "用户邮箱地址"]
    phone: Annotated[str, ..., "用户的手机号"]


agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk @ atguigu.com，手机号：12345678912")
    ]
}
)

for msg in response["messages"]:
    msg.pretty_print()

# 输出模式3：JsonSchema类型

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

json_schema = {
    "title": "ContactInfo",
    "description": "用户的联系方式",
    "type": "object",
    "properties": {
        "name": {
            "description": "用户姓名",
            "type": "string"
        },
        "email": {
            "description": "用户邮箱地址",
            "type": "string"
        },
        "phone": {
            "description": "用户的手机号",
            "type": "string"
        }
    },
    "required": [
        "name",
        "email",
        "phone"
    ]
}
agent = create_agent(
    model=model,
    response_format=ToolStrategy(json_schema)
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：    songhk @ atguigu.com，手机号：12345678912")
        ]
    }
)
for msg in response["messages"]:
    msg.pretty_print()


In [17]:
from typing import Union
from dataclasses import dataclass, Field
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel, Field


@dataclass
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventInfo],
        tool_message_content="已成功抽取信息",
        handle_errors=True

        # handle_errors="请检查输入数据"
    )
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk @ atguigu.com，手机号：12345678912")
        ]
    }
)
for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：songhk @ atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_5ajutslfWihmKzMnDmuQ8857)
 Call ID: call_00_5ajutslfWihmKzMnDmuQ8857
  Args:
    name: 小明
    email: songhk@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

已成功抽取信息


In [14]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：2026年高考报名人数突破1200万")
        ]
    }
)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

从这段话中抽取结构化信息：2026年高考报名人数突破1200万
================================== Ai Message ==================================
Tool Calls:
  EventInfo (call_00_7OEdzrzCcjol4eCoXhLC6048)
 Call ID: call_00_7OEdzrzCcjol4eCoXhLC6048
  Args:
    event_name: 高考报名
    date: 2026年
================================= Tool Message =================================
Name: EventInfo

已成功抽取信息


## 自定义工具消息：tool_message_content参数
